In [ ]:
!pip install captum transformers datasets

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from datasets import load_dataset, Audio
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

import numpy as np

import os
from scipy.interpolate import interp1d
from tqdm import tqdm

### Reference Δείγμα

In [ ]:
TARGET_IDX = 35

### Φόρτωση finetuned μοντέλου

In [ ]:
model_id = "VasilisAsim/data2vec-finetuned-TESS_full"
model_ft = AutoModelForAudioClassification.from_pretrained(model_id)
feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)
model_ft.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model_ft.to(device)
id2label = model_ft.config.id2label

### Φόρτωση dataset (speaker-independent split)

In [ ]:
from datasets import load_dataset, Audio, DatasetDict

def load_tess_speaker_independent():
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))   # για το filter

    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename:
            return "OAF"
        elif "YAF" in filename:
            return "YAF"
        return "unknown"

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")

    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))

    return DatasetDict({"train": train_dataset, "test": test_dataset})

In [ ]:
dataset = load_tess_speaker_independent()
test_dataset = dataset["test"]

### Wrapper για υπολογισμό gradients

In [ ]:
class Data2VecWrapper(nn.Module):
    """
    Wrapper που εκθέτει μόνο τα logits του μοντέλου ως έξοδο ενός forward pass,
    ώστε να μπορεί να κληθεί .backward() πάνω στο επιθυμητό logit για τον
    υπολογισμό Input×Gradient
    """
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, input_values):
        return self.model(input_values=input_values).logits

In [ ]:
wrapped_model = Data2VecWrapper(model_ft).to(device).eval()

### Βοηθητική συναρτήση για την αποθήκευση attribution

In [ ]:
#Resample function
def resample_attribution(attr, target_frames=50):
    attr = np.abs(attr)
    x_original = np.linspace(0, 1, len(attr))
    x_target = np.linspace(0, 1, target_frames)
    f = interp1d(x_original, attr, kind='linear')
    resampled = f(x_target)
    resampled = resampled / (resampled.sum() + 1e-10)
    return resampled

save_dir = "/content/drive/MyDrive/Colab Notebooks/SER_Thesis/01_data2vec_TESS/attributions"
os.makedirs(save_dir, exist_ok=True)

### Εξαγωγή προβλέψεων ανά δείγμα

In [ ]:

def explain_sample_ixg(idx, split="test"):
    """
    Υπολογισμός Input×Gradient attribution για ένα δείγμα. forward pass για
    την προβλεπόμενη κλάση, backward pass πάνω σε εκείνο το logit και
    πολλαπλασιασμός gradient×input
    """
    sample = dataset[split][idx]
    audio = sample["audio"]
    true_label = sample["emotion"]

    input_values = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"], return_tensors="pt"
    ).input_values.to(device)
    input_values.requires_grad_()

    logits = wrapped_model(input_values)
    predicted_id = logits.argmax(dim=-1).item()
    target_logit = logits[0, predicted_id]
    target_logit.backward()

    relevance = (input_values * input_values.grad).squeeze().cpu().detach().numpy()
    return audio["array"], relevance, predicted_id, true_label

In [ ]:
def find_samples_by_class(target_label, split="test"):
    indices = [i for i, label in enumerate(dataset[split]["emotion"]) if label == target_label]
    correct, misclassified = [], []

    for idx in tqdm(indices, desc=f"Επεξεργασία '{target_label}' δειγμάτων"):
        _, _, predicted_id, true_label = explain_sample_ixg(idx, split=split)
        pred_label = id2label[predicted_id]
        if pred_label == true_label:
            correct.append(idx)
        else:
            misclassified.append((idx, pred_label))

    return correct, misclassified

In [ ]:
angry_correct, angry_wrong = find_samples_by_class("angry")
print(f"Σωστά: {len(angry_correct)}, Λάθος: {len(angry_wrong)}")
print(angry_wrong[:5])

In [ ]:
from IPython.display import Audio, display

idx, pred_label = angry_wrong[5]
speech, relevance, predicted_id, true_label = explain_sample_ixg(idx, split="test")

print(f"True label: {true_label} | Predicted: {id2label[predicted_id]} | idx: {idx}")
display(Audio(speech, rate=feature_extractor.sampling_rate))

###Plotting : idx=35 (angry > sad)

In [ ]:
speech, relevance, predicted_id, true_label = explain_sample_ixg(TARGET_IDX, split="test")

relevance_abs = np.abs(relevance)
window = 400
relevance_smooth = np.convolve(relevance_abs, np.ones(window)/window, mode='same')

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(speech, color='gray')
axes[0].set_title(f"Κυματομορφή — True label: {true_label}")

axes[1].plot(relevance_smooth, color='red')
axes[1].set_title(f"Input×Gradient — Predicted: {id2label[predicted_id]}")
plt.tight_layout()
plt.show()

resampled_ig = resample_attribution(relevance, target_frames=50)
np.save(f"{save_dir}/IxG_idx{TARGET_IDX}.npy", resampled_ig)
print(f"Αποθηκεύτηκε: IxG_idx{TARGET_IDX}.npy, shape={resampled_ig.shape}")